# Google Local JSONL → CSV

Notebook, który ulatwia zamianę dowolnego zbioru danych z poniższego linku.
Proces ten zajmuje bardzo dlugo ze względu na wielkość danych.

Dataset: [McAuley Lab — Google Local](https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/#sample-review).

Poniżej znajduję się definicja stalych do przetworzenia danych

Skrypty te powinny być uruchomione jednorazowo na zbiór danych, nie ma konieczności robienia tego ponownie

In [3]:
STATE = 'New_York'

REVIEW_INPUT_PATH =  f"../raw-datasets/review-{STATE}.json.gz"
METADATA_INPUT_PATH = f"../raw-datasets/meta-{STATE}.json.gz"

# Te datasety poniżej, są rozpakowane i przygotowane do analizy EDA oraz przefiltrowania
REVIEW_OUTPUT_PATH = f"../raw-datasets/reviews_{STATE}.csv"
METADATA_OUTPUT_PATH = f"../raw-datasets/metadata_{STATE}.csv"

REVIEW_COLUMNS = ["time", "rating", "text", "gmap_id"]

METADATA_COLUMNS = ["name", "category", "latitude", "longitude", "gmap_id"]


In [4]:
import polars as pl


df = pl.scan_ndjson(METADATA_INPUT_PATH)

df.with_columns(pl.col("category").cast(pl.List(pl.String)).list.join("||")).select(METADATA_COLUMNS).sink_csv(METADATA_OUTPUT_PATH)

In [5]:
import polars as pl


df = pl.scan_ndjson(REVIEW_INPUT_PATH)

df.select(REVIEW_COLUMNS).with_columns(pl.col("text").str.replace_all(r"\n", " ")).sink_csv(REVIEW_OUTPUT_PATH)

## W trakcie analizy danych wybieramy miejsca, które chce brać pod uwagę w analizie.

Dodatkowo kod:

Dla metadanych (zapis `metadata_*_short.csv`): 
- tylko rekordy z danego „boxu” (bbox)

W kolejnym kroku (opinie): 
- wczytuje zapisany plik metadanych i **filtruje po kategoriach** (`CATEGORIES`)
- łączy z opiniami po `gmap_id`, dokleja `name` / `category` / współrzędne, usuwa `gmap_id`
- pomija puste `text`, normalizuje znaki nowej linii w `text`

In [7]:
CATEGORIES = [
    "Park",
    "City park",
    "Playground",
    "Sports complex",
    "Public swimming pool",
    "Recreation center",
    "Train station",
    "Transit station",
    "Bus station",
    "Subway station",
    "Museum",
    "Art gallery",
    "Historical landmark",
    "Monument",
    "Castle",
    "Town square",
    "Plaza",
    "Tourist attraction",
    "Historical place",
    "Shopping mall",
    "Flea market",
    "Farmers' market",
    "Bazaar",
]

SHORTENED_REVIEW_PATH = f"../statics/datasets/reviews_{STATE}_short.csv"
SHORTENED_METADATA_PATH = f"../statics/datasets/metadata_{STATE}_short.csv"

METADATA_INPUT_PATH = f"../raw-datasets/metadata_{STATE}.csv" 
REVIEWS_INPUT_PATH = f"../raw-datasets/reviews_{STATE}.csv"

MIN_LAT = 40.4774 # Southernmost point (Staten Island)
MAX_LAT = 40.9176 # Northernmost point (Bronx)
MIN_LON = -74.2591 # Westernmost point (Staten Island)
MAX_LON = -73.7004 # Easternmost point (Queens)

df_meta_filtered = pl.scan_csv(METADATA_INPUT_PATH, schema_overrides={"gmap_id": pl.String}).filter(
        pl.col("latitude").is_between(MIN_LAT, MAX_LAT) &
        pl.col("longitude").is_between(MIN_LON, MAX_LON)
    ).collect() 

df_meta_filtered.write_csv(SHORTENED_METADATA_PATH)

In [ ]:
df_meta_for_reviews = (
    pl.scan_csv(SHORTENED_METADATA_PATH, schema_overrides={"gmap_id": pl.String})
    .filter(pl.col("category").str.contains("|".join(CATEGORIES)))
    .collect()
)

valid_gmap_ids = df_meta_for_reviews["gmap_id"]

meta_for_join = (
    df_meta_for_reviews.select("gmap_id", "name", "category", "latitude", "longitude")
    .unique(subset=["gmap_id"], keep="any")
    .lazy()
)

(
    pl.scan_csv(
        REVIEWS_INPUT_PATH,
        schema_overrides={"gmap_id": pl.String, "user_id": pl.String},
    )
    .filter(
        pl.col("gmap_id").is_in(valid_gmap_ids.to_list())
        & pl.col("text").is_not_null()
        & (pl.col("text") != "")
    )
    .join(meta_for_join, on="gmap_id", how="inner")
    .with_columns(pl.col("text").str.replace_all(r"\n", " "))
    .drop("gmap_id")
    .sink_csv(SHORTENED_REVIEW_PATH)
)